<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/primeras%20pruebas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIBRERIAS A IMPORTAR
para el correcot funcionamiento y evitar errores

In [1]:
import pandas as pd
import os
from openai import OpenAI
import json
from langsmith import Client
from google.colab import userdata
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


a continuacion declaramos el nombre del archivo el cual posse la los ramos que nos otorgo la institucion

In [2]:
nombre_archivo = 'SAN BERNARDO 2026 002.xlsx'

try:
    df = pd.read_excel(nombre_archivo, sheet_name='BASE')
    print(f" Archivo '{nombre_archivo}' cargado exitosamente.")
    columnas_clave = ['Carrera', 'Jornada', 'Nombre Asignatura', 'Sección', 'Horario', 'Docente']
    display(df[columnas_clave].head())

except FileNotFoundError:
    print(f" Error: No se encontró el archivo '{nombre_archivo}'. Asegúrate de subirlo a Colab.")

 Archivo 'SAN BERNARDO 2026 002.xlsx' cargado exitosamente.


,Carrera,Jornada,Nombre Asignatura,Sección,Horario,Docente
0,INGENIER. EN INFRAESTRUCTURA TECNOLÓGICA,Diurno,CIBERSEGURIDAD OFENSIVA,OCY1104-001D,Lu 8:31:00 - 9:50:00,JOSE ERASMO MATUS
1,INGENIER. EN INFRAESTRUCTURA TECNOLÓGICA,Diurno,CIBERSEGURIDAD OFENSIVA,OCY1104-001D,Lu 10:01:00 - 11:20:00,JOSE ERASMO MATUS
2,INGENIER. EN INFRAESTRUCTURA TECNOLÓGICA,Diurno,CIBERSEGURIDAD EN DESARROLLO,OCY1102-021D,Lu 17:31:00 - 18:50:00,YASSER GUERRA
3,INGENIER. EN INFRAESTRUCTURA TECNOLÓGICA,Diurno,CIBERSEGURIDAD EN DESARROLLO,OCY1102-021D,Lu 16:01:00 - 17:20:00,YASSER GUERRA
4,INGENIER. EN INFRAESTRUCTURA TECNOLÓGICA,Diurno,CIBERSEGURIDAD EN DESARROLLO,OCY1102-001D,Lu 8:31:00 - 9:50:00,JUAN ARTURO SAGARDIA


# CONFIGURACION DE API KEY
en este caso se usa el nombre "OPENAI_API_KEY" en caso de tener tu api key con otro nombre edita el codigo para evitar errores

In [3]:
try:
    llave_groq = userdata.get('OPENAI_API_KEY') #revisa que coinsida el nombre api key
    cliente_ai = OpenAI(
        api_key=llave_groq,
        base_url="https://api.groq.com/openai/v1"
    )
    print("✅ Conectado a GroqCloud exitosamente.")
except Exception as e:
    print(f"Error al cargar la API Key: {e}")

✅ Conectado a GroqCloud exitosamente.


# LECTURA DE LA MALLA CURRICULAR (PDF)
A continuación, instalaremos la librería `pdfplumber` para leer la malla del estudiante,
a continuacion se sube el pdf de tu maya al colab y la variable `nombre_pdf` se le cabia lo que este dentro de las "" por el nombre que posea tu pdf que has subido al colab

In [4]:
!pip install pdfplumber -q
import pdfplumber

nombre_pdf = "maya test.pdf" #a esta variable se le ingrea el mismo nombre que le tenags a la maya que este subida
texto_malla = ""

try:
    with pdfplumber.open(nombre_pdf) as pdf:
        for pagina in pdf.pages:
            texto_malla += pagina.extract_text() + "\n"

    print(f"Malla '{nombre_pdf}' leída exitosamente.")
    print("La IA ya tiene acceso a tus ramos.")
except Exception as e:
    print(f"Error al leer el PDF: {e}. Asegúrate de subir el archivo a Colab.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 828.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 50.0 MB/s eta 0:00:00


Malla 'maya test.pdf' leída exitosamente.
La IA ya tiene acceso a tus ramos.


# CREACIÓN DEL ASISTENTE Y PRIMERA INTERACCIÓN
Aquí unimos la malla extraída. Le daremos instrucciones claras (Prompt) para que actúe como un consejero académico de Duoc UC y analice el semestre del alumno.

In [6]:
instrucciones_sistema = f"""
Eres un asistente académico experto de Duoc UC. A continuación te entregaré el texto extraído
de la malla curricular del estudiante.

Tu tarea es iniciar una conversación con el estudiante para ayudarlo a planificar su horario.
Sigue estrictamente estos pasos:
1. Saluda al estudiante con entusiasmo y menciónale que ya tienes su malla curricular cargada.
2. Pregúntale en qué semestre se va a matricular.
3. Pregúntale si tiene algún ramo atrasado que deba priorizar.
4. Pregúntale si prefiere estudiar en la mañana (diurno) o en la tarde (vespertino).

Aquí está la malla del estudiante:
{texto_malla}
"""

# 3 simular pregunta
mensaje_usuario = "Hola, necesito ayuda para ver qué ramos tomar este semestre."
print(f"\n👤 Estudiante: {mensaje_usuario}\n")
print("-" * 50)


try:
    respuesta = cliente_ai.chat.completions.create(
        model="openai/gpt-oss-120b", # El modelo usado
        messages=[
            {"role": "system", "content": instrucciones_sistema},
            {"role": "user", "content": mensaje_usuario}
        ],
        temperature=0.7
    )

    print("🤖 Asistente DuocBot:")
    print(respuesta.choices[0].message.content)

except Exception as e:
    print(f"Error al generar respuesta: {e}")


👤 Estudiante: Hola, necesito ayuda para ver qué ramos tomar este semestre.

--------------------------------------------------
🤖 Asistente DuocBot:
¡Hola! 👋 ¡Qué gusto saludarte! Ya tengo cargada tu malla curricular de Ingeniería en Informática – Mención Inteligencia Artificial, así que estoy listo para ayudarte a armar tu horario.

1️⃣ ¿En qué semestre vas a matricularte este año?  
2️⃣ ¿Tienes algún ramo atrasado o pendiente que necesites priorizar?  
3️⃣ ¿Prefieres estudiar en la mañana (diurno) o en la tarde (vespertino)?  

¡Con esa información podré recomendarte los cursos que mejor se ajusten a tus necesidades!


# CHARLA DE PRUEBA CON EL ASISTENTE


In [7]:
historial_mensajes = [
    {"role": "system", "content": instrucciones_sistema},
    {"role": "user", "content": mensaje_usuario},
    {"role": "assistant", "content": respuesta.choices[0].message.content}
]

print("Modo Chat Activado. (Escribe 'salir' si quieres detener la conversación)\n")

while True:

    nueva_respuesta = input("👤 Tú: ")

    if nueva_respuesta.lower() == 'salir':
        print("\n👋 Chat finalizado.")
        break

    print("-" * 50)


    historial_mensajes.append({"role": "user", "content": nueva_respuesta})


    try:
        nueva_respuesta_ai = cliente_ai.chat.completions.create(
            model="openai/gpt-oss-120b", # El modelo usado
            messages=historial_mensajes,
            temperature=0.7
        )

        texto_ia = nueva_respuesta_ai.choices[0].message.content
        print(f"\n🤖: {texto_ia}\n")


        historial_mensajes.append({"role": "assistant", "content": texto_ia})

    except Exception as e:
        print(f"\n Error en el chat: {e}")
        break

Modo Chat Activado. (Escribe 'salir' si quieres detener la conversación)



KeyboardInterrupt: Interrupted by user